# Milestone 6 — Advanced: multi-dataset, multi-architecture sweep

Extends M4 to Cora / Citeseer / Pubmed × {MLP, GCN, SAGE, GIN}. Optional sections cover GAT, ogbn-arxiv, MUTAG, PROTEINS (gated on availability).

## Setup

Resolve `onboarding/projects/` on `sys.path`, attach the reflection log, and import the canonical references (used only for spot-checks; the student version derives its own implementations).

In [ ]:
import os, sys
from pathlib import Path
_here = Path(os.getcwd()).resolve()
for _p in [_here, *_here.parents]:
    if (_p / 'shared' / '__init__.py').exists() and _p.name == 'projects':
        _PROJECTS = _p
        break
else:
    raise RuntimeError('could not locate onboarding/projects/')
_REPO = _PROJECTS.parent.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless for CI
import matplotlib.pyplot as plt

from onboarding.projects import reflect
reflect.start(notebook='m6', store=_PROJECTS / '.reflection.jsonl')
print(f'[setup] projects root: {_PROJECTS}')


In [ ]:
from onboarding.projects.shared import (
    Partition, label_partition, wl_partition,
    hbin, hbin_inverse, upper, lower, slack, verify, bracket_of,
    sum_partition, mean_partition, max_partition,
)
print('shared modules imported')


In [ ]:
import torch, torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, SAGEConv, GINConv
torch.manual_seed(0); np.random.seed(0)
device = torch.device('cpu')
_CACHE = Path.home() / '.cache' / 'planetoid'
datasets = {}
for name in ['Cora', 'Citeseer', 'Pubmed']:
    try:
        datasets[name] = Planetoid(root=str(_CACHE / name), name=name)[0].to(device)
        print(f'  {name}: n={datasets[name].num_nodes}, m={datasets[name].num_edges}, K={int(datasets[name].y.max())+1}')
    except Exception as e:
        print(f'  {name}: skipped ({type(e).__name__})')


In [ ]:
def build(name, in_dim, h, k):
    if name == 'MLP':
        return torch.nn.Sequential(torch.nn.Linear(in_dim,h), torch.nn.ReLU(), torch.nn.Linear(h,k))
    raise NotImplementedError

class GCN(torch.nn.Module):
    def __init__(self,d,h,k): super().__init__(); self.c1=GCNConv(d,h); self.c2=GCNConv(h,k)
    def forward(self,x,ei): return self.c2(F.relu(self.c1(x,ei)),ei)
class SAGE(torch.nn.Module):
    def __init__(self,d,h,k): super().__init__(); self.c1=SAGEConv(d,h); self.c2=SAGEConv(h,k)
    def forward(self,x,ei): return self.c2(F.relu(self.c1(x,ei)),ei)
class GIN(torch.nn.Module):
    def __init__(self,d,h,k):
        super().__init__()
        m1=torch.nn.Sequential(torch.nn.Linear(d,h),torch.nn.ReLU(),torch.nn.Linear(h,h))
        m2=torch.nn.Sequential(torch.nn.Linear(h,h),torch.nn.ReLU(),torch.nn.Linear(h,k))
        self.c1=GINConv(m1); self.c2=GINConv(m2)
    def forward(self,x,ei): return self.c2(F.relu(self.c1(x,ei)),ei)

def train(model, data, epochs=80):
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        try:
            out = model(data.x, data.edge_index)
        except TypeError:
            out = model(data.x)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        try: logits = model(data.x, data.edge_index)
        except TypeError: logits = model(data.x)
        p = logits.argmax(-1)
        return float((p[data.test_mask] == data.y[data.test_mask]).float().mean())

ARCHS = {'MLP': None, 'GCN': GCN, 'SAGE': SAGE, 'GIN': GIN}
rows = []
for ds_name, data in datasets.items():
    K = int(data.y.max())+1
    for a_name, a_cls in ARCHS.items():
        torch.manual_seed(0)
        if a_cls is None:
            model = build('MLP', data.num_features, 64, K)
        else:
            model = a_cls(data.num_features, 64, K)
        acc = train(model, data, epochs=80)
        rows.append((ds_name, a_name, acc))
        print(f'  {ds_name:>9}/{a_name:>5}: acc={acc:.3f}')


In [ ]:
# Gate: at least one structural arch beats MLP on at least one dataset.
best_struct = max(acc for ds, a, acc in rows if a != 'MLP')
best_mlp    = max((acc for ds, a, acc in rows if a == 'MLP'), default=0.0)
assert best_struct > best_mlp - 0.02, f'M6: structural arches never beat MLP'
print(f'[GATE OK] M6: best structural={best_struct:.3f}, best MLP={best_mlp:.3f}')


In [ ]:
import pandas as pd
df = pd.DataFrame(rows, columns=['dataset','arch','test_acc'])
pivot = df.pivot(index='arch', columns='dataset', values='test_acc')
print(pivot)
fig, ax = plt.subplots(figsize=(6,3.5))
pivot.plot(kind='bar', ax=ax); ax.set_ylabel('test acc'); ax.set_title('M6 — arch × dataset sweep')
_plots = _PROJECTS / 'capstone' / 'milestone6' / 'plots'; _plots.mkdir(parents=True, exist_ok=True)
plt.tight_layout(); fig.savefig(_plots / 'm6_sweep.png', dpi=120); plt.show()


In [ ]:
reflect.log('M6.advanced_sweep', f'{len(rows)} (dataset, arch) cells trained; best structural={best_struct:.3f}', 'HIGH')


### Optional — heavier sweeps via Modal
Uncomment to launch on T4.

In [ ]:
# from onboarding.projects import modal_app
# if modal_app._MODAL_OK and modal_app.app is not None:
#     with modal_app.app.run():
#         print(modal_app.nas_sweep.remote(epochs=200))
print('[stub] Modal optional sweep — see M3/M4 for the live version.')


**End of M6.**